In [ ]:
import pandas as pd
import requests
import time

# ==========================================
# 1. LOAD EXCEL FILE
# ==========================================
print("1. Loading Excel file...")

df_stops = pd.read_excel("BusStops.xlsx")

df_stops['Coordinates'] = (
    df_stops['Coordinates']
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

df_stops[['Longitude', 'Latitude']] = (
    df_stops['Coordinates']
    .str.split(',', expand=True)
)

df_stops['Latitude'] = pd.to_numeric(
    df_stops['Latitude'].str.strip(),
    errors='coerce'
)

df_stops['Longitude'] = pd.to_numeric(
    df_stops['Longitude'].str.strip(),
    errors='coerce'
)

print(f"-> Successfully loaded {len(df_stops)} bus stops")


# ==========================================
# 2. REAL WEATHER DATA
# ==========================================
print("2. Loading REAL weather data...")

WEATHER_API_KEY = "IDE_IRD_AZ_OPENWEATHER_API_KULCSOD"

temperatures = []
humidities = []

weather_cache = {}

def get_weather(lat, lon):

    key = (round(lat, 3), round(lon, 3))

    if key in weather_cache:
        return weather_cache[key]

    url = (
        f"https://api.openweathermap.org/data/2.5/weather"
        f"?lat={lat}"
        f"&lon={lon}"
        f"&appid={WEATHER_API_KEY}"
        f"&units=metric"
    )

    try:

        response = requests.get(url, timeout=10)

        data = response.json()

        temp = data['main']['temp']
        humidity = data['main']['humidity']

        weather_cache[key] = (temp, humidity)

        return temp, humidity

    except Exception as e:

        print("Weather API error:", e)

        weather_cache[key] = (None, None)

        return None, None


for index, row in df_stops.iterrows():

    if index % 50 == 0:
        print(f"Weather processed: {index}/{len(df_stops)}")

    lat = row['Latitude']
    lon = row['Longitude']

    if pd.isna(lat) or pd.isna(lon):

        temperatures.append(None)
        humidities.append(None)

        continue

    temp, hum = get_weather(lat, lon)

    temperatures.append(temp)
    humidities.append(hum)

    time.sleep(0.1)

df_stops['Temperature'] = temperatures
df_stops['Humidity'] = humidities


# ==========================================
# 3. DOWNLOAD OSM DATA
# ==========================================
print("3. Downloading REAL OSM vegetation and benches...")

OSM_URL = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:40];
(
  node["natural"="tree"](47.45,21.50,47.60,21.72);
  node["amenity"="bench"](47.45,21.50,47.60,21.72);
);
out body;
"""

trees_data = []
benches_data = []

try:

    response = requests.post(
        OSM_URL,
        data={'data': query},
        headers={'User-Agent': 'BusStopProject/1.0'},
        timeout=60
    )

    data = response.json()

    elements = data.get("elements", [])

    print(f"-> Downloaded {len(elements)} OSM elements")

    for e in elements:

        lat = e.get("lat")
        lon = e.get("lon")

        tags = e.get("tags", {})

        if lat is None or lon is None:
            continue

        if tags.get("natural") == "tree":
            trees_data.append((lat, lon))

        if tags.get("amenity") == "bench":
            benches_data.append((lat, lon))

except Exception as e:

    print("OSM ERROR:", e)

print(f"-> Trees loaded: {len(trees_data)}")
print(f"-> Benches loaded: {len(benches_data)}")


# ==========================================
# 4. PROCESS BUS STOPS
# ==========================================
print("4. Processing bus stops...")

vegetations = []
spaces = []

RADIUS = 0.00035

for index, row in df_stops.iterrows():

    if index % 50 == 0:
        print(f"Processed {index}/{len(df_stops)}")

    lat = row['Latitude']
    lon = row['Longitude']

    if pd.isna(lat) or pd.isna(lon):

        vegetations.append("No coordinates")
        spaces.append("No coordinates")

        continue

    tree_count = 0
    bench_count = 0

for t_lat, t_lon in trees_data:

    if abs(t_lat - lat) < RADIUS and abs(t_lon - lon) < RADIUS:
        tree_count += 1

for b_lat, b_lon in benches_data:

    if abs(b_lat - lat) < RADIUS and abs(b_lon - lon) < RADIUS:
        bench_count += 1

vegetations.append(tree_count)

spaces.append(bench_count * 3)

df_stops['Vegetation'] = vegetations
df_stops['Spaces available'] = spaces


# ==========================================
# 5. GENERATE ISSUES
# ==========================================
print("5. Generating issue reports...")

problems = []

for _, row in df_stops.iterrows():

    issues = []

    if str(row['Covered']).strip().lower() == 'no':
        issues.append("No shelter")

    if str(row['Lightning']).strip().lower() == 'no':
        issues.append("No street lighting")

    if str(row['Wheelchair accessible']).strip().lower() == 'no':
        issues.append("Not wheelchair accessible")

    if str(row['Bus bay available']).strip().lower() == 'no':
        issues.append("No bus bay available")

    if issues:
        problems.append(", ".join(issues))
    else:
        problems.append("No reported issues")

df_stops['Problems'] = problems


# ==========================================
# 6. CLEANUP
# ==========================================
df_final = df_stops.drop(
    columns=['Latitude', 'Longitude']
)


# ==========================================
# 7. SAVE JSON
# ==========================================
output_file = "BusStop.json"

df_final.to_json(
    output_file,
    orient="records",
    force_ascii=False,
    indent=4
)

print("\n🎉 Program finished successfully!")
print(f"Output file: {output_file}")


# ==========================================
# 8. PREVIEW
# ==========================================
print("\nFirst 5 rows:")

print(
    df_final[
        [
            'Bus stop',
            'Vegetation',
            'Temperature',
            'Humidity',
            'Spaces available',
            'Problems'
        ]
    ].head()
)

1. Loading Excel file...
-> Successfully loaded 687 bus stops
2. Generating weather data...
3. Downloading REAL OSM vegetation and benches...
-> Downloaded 1250 OSM elements
-> Trees loaded: 832
-> Benches loaded: 418
4. Processing bus stops...
Processed 0/687
Processed 50/687
Processed 100/687
Processed 150/687
Processed 200/687
Processed 250/687
Processed 300/687
Processed 350/687
Processed 400/687
Processed 450/687
Processed 500/687
Processed 550/687
Processed 600/687
Processed 650/687
5. Generating issue reports...

🎉 Program finished successfully!
Output file: BusStops_FELTOLTOTT.json

First 5 rows:
         Bus stop      Vegetation  Temperature  Humidity   Spaces available  \
0  Leiningen utca  0 trees nearby         20.2        59  0 seats available   
1  Leiningen utca  0 trees nearby         22.4        37  0 seats available   
2    Somlyai utca  0 trees nearby         20.7        39  0 seats available   
3    Somlyai utca  0 trees nearby         20.0        41  0 seats availa